In [ ]:
!pip install transformers datasets peft bitsandbytes accelerate trl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/qlora-matrimonial/model"

In [ ]:
import json, os
from datasets import Dataset
from transformers import AutoTokenizer
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
from peft import PeftModel



from datetime import datetime, time
import logging
from logging import basicConfig, getLogger, INFO
import zipfile
from logging.handlers import RotatingFileHandler

class ZippedRotatingFileHandler(RotatingFileHandler):
    def doRollover(self):
        super().doRollover()
        # The 'old' file is now at baseFilename.1
        old_log = self.baseFilename + ".1"
        if os.path.exists(old_log):
            with zipfile.ZipFile(f"{old_log}.zip", 'w', zipfile.ZIP_DEFLATED) as zf:
                zf.write(old_log, os.path.basename(old_log))
            os.remove(old_log)

# Setup
handler = ZippedRotatingFileHandler("peft_trainer.log", maxBytes=1024*1024, backupCount=5)
logging.basicConfig(handlers=[handler], level=logging.INFO)
logger = logging.getLogger("peft_trainer")

PROFILE_DATA_PATH = "/content/drive/MyDrive/qlora-matrimonial/training-data"

# Function to load multiple JSON files and combine them into a single Hugging Face Dataset
def load_json_files(file_paths):
    all_samples = []

    for fp in file_paths:
        with open(fp, "r") as f:
            data = json.load(f)
            all_samples.extend(data)

    return Dataset.from_list([{"messages": sample} for sample in all_samples])

def get_training_data_files(profile_data_version, filter_name):
        #Go through all available files in the dir.
        #collect all such ids.

        directory_to_scan = f'{PROFILE_DATA_PATH}' 

        data_files = []

        try:
            for filename in os.listdir(directory_to_scan):
                full_path = os.path.join(directory_to_scan, filename)
                if ".json" in full_path and os.path.isfile(full_path):
                    data_files.append(full_path)
        except Exception as e:
            logger.error(f"Error while scanning directory {directory_to_scan}: {e}")
        return data_files

In [22]:

#MODEL_NAME = "meta-llama/Meta-Llama-3.2-3B-Instruct"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
#trying to further tune a model already fine tuned,
#MODEL_NAME = "././qlora-llama3-matrimonial-v2/merged_model"


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [23]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

In [35]:
def get_max_length(dataset, sample_size=200):
    lengths = []

    for i in range(min(sample_size, len(dataset))):
        text = dataset[i]["text"]
        tokens = tokenizer(text)["input_ids"]
        if len(tokens) > 1500:
            print(f"Got a training input with token length:{len(tokens)}, text length:{len(text)}")
            if len(text) > 7900:
                print(f"SUPER LONG Sample:{text}")
        
        lengths.append(len(tokens))

    print(f"Max length (sampled): {max(lengths)}")
    print(f"Avg length: {sum(lengths)/len(lengths):.2f}")

    #return int(min(max(lengths) * 1.1, 4096))  # add buffer
    #return int(max(lengths) * 1.1)  # add buffer, no hard limit for now
    return int(max(lengths))

In [25]:
def tokenize_with_mask(example):
    messages = example["messages"]

    input_ids = []
    labels = []

    for i, msg in enumerate(messages):
        role = msg["role"]

        text = tokenizer.apply_chat_template(
            [msg],
            tokenize=False,
            add_generation_prompt=False
        )

        tokens = tokenizer(text, add_special_tokens=False)["input_ids"]

        input_ids.extend(tokens)

        if role == "assistant":
            labels.extend(tokens)
        else:
            labels.extend([-100] * len(tokens))

    return {
        "input_ids": input_ids,
        "labels": labels
    }

In [36]:
#file_paths = ["data1.json", "data2.json"]
file_paths = get_training_data_files("v1", "peft")
print(f"Found {len(file_paths)} files for training.")
ds = load_json_files(file_paths)

# (optional) inspect
print(ds[0])

# format text (for length calc)
ds = ds.map(format_chat)

max_seq_len = get_max_length(ds)
#TODO: HARDCODED MAX SEQ LEN FOR NOW, CAN BE DYNAMIC BASED ON DATA
max_seq_len = min(max_seq_len, 4096)

# re-tokenize properly
ds = load_json_files(file_paths)

ds = ds.map(tokenize_with_mask)

# truncate + pad
def pad_truncate(example):
    input_ids = example["input_ids"][:max_seq_len]
    labels = example["labels"][:max_seq_len]

    pad_len = max_seq_len - len(input_ids)

    input_ids += [tokenizer.pad_token_id] * pad_len
    labels += [-100] * pad_len

    attention_mask = [1 if t != tokenizer.pad_token_id else 0 for t in input_ids]

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

ds = ds.map(pad_truncate)
ds = ds.train_test_split(test_size=0.1)

Found 12 files for training.
{'messages': [{'content': "\nYou are a matrimonial assistant. Analyse the given Boy and Girl profiles for matrimonial compatibility using instructions:\n\n    1. You are doing the analysis from Girl's point of view.\n    2. Under Basic details level one header, include\n        2.a. Boy's Full Name\n        2.b. Girl's Full Name.\n        2.c. Boy's Profile Picture with appropriate mark up tag with size 200x200. Mark Up Syntax: ![<Full Name>](<Image URL>)\n    \n", 'role': 'system'}, {'content': "\nUse the JSON profiles attached.\nBoy profile: {'Full Name': 'Nalin Milind Kulkarni', 'Photo 1': 'https://res.cloudinary.com/wiwaha/image/upload/t_profile_view/AnuroopVar/6212025113442AM_335627_IMG_4745.jpg'}\nGirl profile: {'Full Name': 'Mrunal Kulkarni'}\nSTRICT OUTPUT FORMAT: \n# Basic Details (Girl’s POV)\n\n- **Boy’s Full Name:** <boy's full name here>  \n- **Girl’s Full Name:** <girl's full name here> \n- **Boy’s Profile Picture (200x200):**  \n\n  ![<boy's 

Map: 100%|██████████| 368/368 [00:00<00:00, 7886.16 examples/s]


Got a training input with token length:1652, text length:6790
Got a training input with token length:1787, text length:7152
Got a training input with token length:1745, text length:7064
Got a training input with token length:1817, text length:7272
Got a training input with token length:1775, text length:7127
Got a training input with token length:1765, text length:7215
Got a training input with token length:1792, text length:7265
Got a training input with token length:1788, text length:7187
Got a training input with token length:1695, text length:6958
Got a training input with token length:2000, text length:7963
SUPER LONG Sample:<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 18 Apr 2026

You are a matrimonial assistant. Analyse the expectations mentioned in the Girl's profile and cross-check them against the corresponding information in the Boy's profile.
1. For each expectation mentioned in the Girl's profile, check the 

Map: 100%|██████████| 368/368 [00:00<00:00, 815.08 examples/s] 


In [ ]:


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config, #For mac
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True

    #device_map={"": "cpu"},

)

Loading weights: 100%|██████████| 254/254 [00:07<00:00, 33.63it/s]


In [28]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    #target_modules=["q_proj", "v_proj"],  # can expand later
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [29]:
print(ds["train"].column_names)

['messages', 'input_ids', 'labels', 'attention_mask']


In [30]:
ds = ds.remove_columns(
    [col for col in ds["train"].column_names if col not in ["input_ids", "labels", "attention_mask"]]
)

In [ ]:
from transformers import default_data_collator

#TODO: HARDCODED TRAINING PARAMS FOR NOW, CAN BE TUNED LATER
per_device_train_batch_size = 2
gradient_accumulation_steps = 4
fp16 = True
num_train_epochs = 3

training_args = TrainingArguments(
    output_dir="./qlora-llama3-matrimonial",
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    learning_rate=2e-4,
    logging_steps=10,

    save_strategy="epoch",      # ✅ save every epoch
    save_total_limit=2,         # keep last 2 checkpoints

    #evaluation_strategy="epoch",
    fp16=fp16,
    report_to="none",
    disable_tqdm=False,
    remove_unused_columns=False,

    optim="adamw_torch", #For now, using PyTorch's native AdamW which is compatible with 4-bit training.

    per_device_eval_batch_size=2,
    weight_decay=0.01,
    warmup_ratio=0.06,
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=50,
    #seed=cfg.seed,
    #optim="paged_adamw_8bit",
    #dataloader_drop_last=True,
    

)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    #tokenizer=tokenizer,
    data_collator=default_data_collator,
    processing_class=None
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [32]:
batch = next(iter(trainer.get_train_dataloader()))
print(batch.keys())

dict_keys(['input_ids', 'labels', 'attention_mask'])


/Users/skulkarn/anaconda3/envs/peft_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [33]:
sample = ds["train"][0]

assert "input_ids" in sample
assert "labels" in sample
assert "attention_mask" in sample

In [34]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


RuntimeError: MPS backend out of memory (MPS allocated: 47.28 GiB, other allocations: 458.12 MiB, max allowed: 47.74 GiB). Tried to allocate 23.44 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [15]:
# Save PEFT adapter
trainer.model.save_pretrained("./qlora-llama3-matrimonial/final_adapter")
tokenizer.save_pretrained("./qlora-llama3-matrimonial/final_adapter")

('./qlora-llama3-matrimonial/final_adapter/tokenizer_config.json',
 './qlora-llama3-matrimonial/final_adapter/chat_template.jinja',
 './qlora-llama3-matrimonial/final_adapter/tokenizer.json')

In [16]:


base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

peft_model = PeftModel.from_pretrained(
    base_model,
    "./qlora-llama3-matrimonial/final_adapter"
)

# 🔥 Merge weights
merged_model = peft_model.merge_and_unload()

Loading weights: 100%|██████████| 254/254 [00:10<00:00, 23.24it/s]


In [17]:
merged_model.save_pretrained("./qlora-llama3-matrimonial/merged_model")
tokenizer.save_pretrained("./qlora-llama3-matrimonial/merged_model")

Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.93s/it]


('./qlora-llama3-matrimonial/merged_model/tokenizer_config.json',
 './qlora-llama3-matrimonial/merged_model/chat_template.jinja',
 './qlora-llama3-matrimonial/merged_model/tokenizer.json')